# Learned reasoning, full fine-tuning: GRPO with fp32 master weights
### Qwen2.5 0.5B · 1.5B, every weight trainable and able to move

| § | Concept | Figure |
|---|---|---|
| 1 | **The loop** — sample → verify → reinforce; the group is its own baseline | GRPO advantages on real prompts, no critic |
| 2 | **Axis 1: RL lifts the curve** | training reward and trace length |
| 2b | **Did the weights move?** | share of weights changed and update size, per checkpoint and layer type |
| 3 | **Axis 2: thinking slides along it** | maj@k vs. log(k) and accuracy vs. token budget, per checkpoint |
| 4 | **Both axes** | substitution heatmap |
| 5 | **Stacking on top** | base@1 → RL@1 → maj@k → pass@k |
| 6 | **Model size picks the curve** | overlay of the sizes in this run |

- **fp32 master weights, bf16 autocast.** The v3 run loaded the model in bf16, so the optimizer updated bf16 weights. A 3e-6 Adam step is far below bf16's spacing (≈1e-4 near |w| = 0.03), so most updates rounded to zero: after 400 steps only 1.7% of the 1.5B's weights differed from the base (§2b's measurement, run on that checkpoint). §2b measures this run the same way.
- **Everything else is v3:** lr, 16 prompts × 8 samples per step, the difficulty filter, the DAPO loss, 400 steps and the eval sweep. A change in the curves is the precision fix. The 1.5B's micro-batch is halved and its accumulation doubled to fit the fp32 optimizer state; prompts per step are unchanged.
- **Checkpoints are saved in bf16**, which is what vLLM serves and what the distillation notebook reads.
- **Sizes.** 0.5B and 1.5B, both full fine-tuning. The 1.5B needs an 80 GB card (fp32 weights, gradients and Adam ≈ 25 GB beside vLLM); the 0.5B also runs on 40 GB. The 0.5B is GRPO straight on the distillation notebook's student.
- **Every run is full.** Each run gets its own `run-<UTC stamp>` folder; a disconnect resumes the unfinished run, and a finished run (one with `derived.json`) is never read again.

**Time on an A100 80GB (screen + RL + eval):** 1.5B ≈ 2–2.5 h · 0.5B ≈ 1 h.

> **Run order.** Cell 0.1 installs and then **restarts the runtime on purpose**. Afterwards run all cells from the top; the install cell skips in a second.

## 0. Environment

### 0.1 Install (idempotent; restarts the runtime once)
Do **not** pin `torch` by hand in Colab. vLLM pins the exact torch it was built against and pulls the matching `nvidia-*` CUDA libraries; installing torch separately is what produces `libtorch_cuda.so: undefined symbol: ncclCommResume`. Order: vLLM first, then the libraries that don't touch torch.

Last verified on Colab A100 80GB / Python 3.13 with: torch 2.13.0+cu130 · vLLM 0.32.0 · TRL 1.17.1 · transformers 5.19.0.

In [ ]:
import os, sys, subprocess, importlib.metadata as md
MARK = "/content/.two_axis_env_ok"

def pip(*args):
    print(">", "pip", *args); subprocess.run([sys.executable, "-m", "pip", *args], check=True)

if not os.path.exists(MARK):
    pip("install", "-q", "--upgrade", "pip")
    pip("uninstall", "-y", "-q", "torchvision", "torchaudio")        # Colab's copies will mismatch vLLM's torch; nothing here needs them
    pip("install", "-q", "--upgrade", "vllm>=0.8.5,<0.33.0")         # brings its own torch + nvidia-* libs, all mutually consistent
    pip("install", "-q", "--upgrade", "trl>=0.19", "peft", "datasets", "accelerate", "pandas", "matplotlib")
    # --- self-check in a fresh interpreter; repair the one classic failure (NCCL wheel not matching torch) ---
    chk = subprocess.run([sys.executable, "-c", "import torch, vllm, trl; print(torch.__version__, vllm.__version__, trl.__version__)"],
                         capture_output=True, text=True)
    if chk.returncode != 0 and "nccl" in chk.stderr.lower():
        need = [r for r in md.requires("torch") if r.startswith("nvidia-nccl")][0].split(";")[0].strip()
        print("repairing NCCL ->", need); pip("install", "-q", "--force-reinstall", "--no-deps", need)
        chk = subprocess.run([sys.executable, "-c", "import torch, vllm, trl"], capture_output=True, text=True)
    if chk.returncode != 0:
        print(chk.stderr[-3000:]); raise SystemExit("environment check failed — see error above")
    print("versions:", chk.stdout.strip())
    open(MARK, "w").write("ok")
    print("\nEnvironment ready. Restarting the runtime now (expected). Then run all cells from the top.")
    os.kill(os.getpid(), 9)
else:
    print("environment already prepared — skipping install")

### 0.2 Imports and GPU

In [ ]:
import os, sys, json, gc, glob, shlex, subprocess, random, time
from datetime import datetime, timezone
from pathlib import Path
from dataclasses import dataclass, asdict, field
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import trl, vllm, transformers

gpu = torch.cuda.get_device_properties(0); VRAM_GB = round(gpu.total_memory / 2**30)
print(f"GPU: {gpu.name} | VRAM {VRAM_GB} GB | torch {torch.__version__} cu{torch.version.cuda} | vllm {vllm.__version__} | trl {trl.__version__} | transformers {transformers.__version__}")
assert torch.cuda.is_available()
# optional: silence the HF Hub rate-limit warning and speed up downloads
# from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

### 0.3 Knobs — the only cell you edit
`RUN_SIZES` is processed in order; each size gets its own folder inside this run's folder, and finished stages are skipped on re-run.

In [ ]:
SEED           = 42
RUN_SIZES      = ["1.5B"]          # any of "0.5B", "1.5B", in the order to run
GPU_PRESET     = "auto"            # "auto" | "A100-40GB" | "A100-80GB" | "H100-80GB"
USE_VLLM       = True
MOUNT_DRIVE    = True
MASTER_DTYPE   = "float32"         # the weights the optimizer updates; the matmuls still run in bf16 autocast
FILTER_PROMPTS = True              # train only on prompts the base model solves sometimes but not always (0 < pass rate <= FILTER_MAX)
FILTER_MAX     = 0.75
FILTER_N       = 4000
LOSS_TYPE      = "dapo"
RESULTS_ROOT   = "/content/drive/MyDrive/two_axis" if MOUNT_DRIVE else "/content/two_axis"

# Per-model settings for an 80 GB card. bsz counts completions per micro-batch; prompts/step = bsz*accum/G = 16 for both.
MODEL_PRESETS = {
    "0.5B": dict(model_id="Qwen/Qwen2.5-0.5B-Instruct", lr=3e-6, G=8, bsz=32, accum=4, comp_len=640, vllm_util=0.25,
                 rl_steps=400, n_eval=600, k_max=32, budgets=[64,128,256,512,1024], min_vram=36),
    "1.5B": dict(model_id="Qwen/Qwen2.5-1.5B-Instruct", lr=3e-6, G=8, bsz=16, accum=8, comp_len=640, vllm_util=0.30,
                 rl_steps=400, n_eval=600, k_max=32, budgets=[64,128,256,512,1024], min_vram=70),
}
GPU_SCALE = {   # applied on top of the 80 GB presets
    "A100-40GB": dict(bsz_div=2, accum_mul=2, comp_len=512, n_eval_mul=0.67, k_div=2),
    "A100-80GB": dict(bsz_div=1, accum_mul=1, comp_len=None, n_eval_mul=1.0, k_div=1),
    "H100-80GB": dict(bsz_div=1, accum_mul=1, comp_len=768, n_eval_mul=1.33, k_div=1),
}
def pick_gpu_preset():
    if GPU_PRESET != "auto": return GPU_PRESET
    if VRAM_GB >= 70: return "H100-80GB" if "H100" in gpu.name else "A100-80GB"
    if VRAM_GB >= 36: return "A100-40GB"
    raise SystemExit(f"{VRAM_GB} GB is below the 40 GB floor")

if MOUNT_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
KNOBS = dict(SEED=SEED, RUN_SIZES=RUN_SIZES, gpu_preset=pick_gpu_preset(), USE_VLLM=USE_VLLM, MASTER_DTYPE=MASTER_DTYPE,
             FILTER_PROMPTS=FILTER_PROMPTS, FILTER_MAX=FILTER_MAX, FILTER_N=FILTER_N, LOSS_TYPE=LOSS_TYPE, MODEL_PRESETS=MODEL_PRESETS)
ROOT = Path(RESULTS_ROOT) / "grpo_full_ft"; ROOT.mkdir(parents=True, exist_ok=True)
unfinished = sorted(p for p in ROOT.glob("run-*") if not (p / "derived.json").exists())
if unfinished:
    RUN = unfinished[-1]; saved = json.load(open(RUN / "knobs.json"))
    changed = {k: (saved.get(k), v) for k, v in json.loads(json.dumps(KNOBS)).items() if saved.get(k) != v}
    assert not changed, f"{RUN.name} is unfinished but was started with other knobs {changed}: restore them, or rename that folder to start a new run"
    print(f"resuming unfinished {RUN}")
else:
    RUN = ROOT / f"run-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}"; RUN.mkdir()
    json.dump(KNOBS, open(RUN / "knobs.json", "w"), indent=1); print(f"new run {RUN}")

@dataclass
class Config:
    model_size: str
    seed: int = SEED
    gpu_preset: str = field(default_factory=pick_gpu_preset)
    use_vllm: bool = USE_VLLM
    master_dtype: str = MASTER_DTYPE
    model_id: str = ""
    learning_rate: float = 0.0; num_generations: int = 0; per_device_bsz: int = 0; grad_accum: int = 0
    max_prompt_len: int = 256; max_completion_len: int = 0; vllm_util: float = 0.0; rl_steps: int = 0
    temperature: float = 1.0; beta: float = 0.0; loss_type: str = LOSS_TYPE
    filter_prompts: bool = FILTER_PROMPTS; filter_max: float = FILTER_MAX; filter_n: int = FILTER_N
    n_eval: int = 0; k_max: int = 0; eval_max_tokens: int = 0; eval_temperature: float = 0.7
    budgets: list = field(default_factory=list)
    run_dir: str = ""; ckpt_dir: str = ""

    def __post_init__(self):
        m, s = MODEL_PRESETS[self.model_size], GPU_SCALE[self.gpu_preset]
        assert VRAM_GB >= m["min_vram"], f"{self.model_size} full fine-tuning needs >= {m['min_vram']} GB (have {VRAM_GB})"
        self.model_id, self.learning_rate, self.num_generations = m["model_id"], m["lr"], m["G"]
        self.per_device_bsz = max(m["G"], m["bsz"] // s["bsz_div"]); self.grad_accum = m["accum"] * s["accum_mul"]
        self.max_completion_len = s["comp_len"] or m["comp_len"]; self.vllm_util, self.rl_steps = m["vllm_util"], m["rl_steps"]
        self.n_eval = int(m["n_eval"] * s["n_eval_mul"]); self.k_max = max(4, m["k_max"] // s["k_div"])
        self.budgets = list(m["budgets"]); self.eval_max_tokens = max(self.budgets)
        self.run_dir = f"{RUN}/{self.model_size}"; self.ckpt_dir = f"{self.run_dir}/ckpt"
        assert (self.per_device_bsz * self.grad_accum) % self.num_generations == 0

    @property
    def checkpoint_steps(self):
        steps, s = [0], max(10, self.rl_steps // 16)
        while s < self.rl_steps: steps.append(s); s *= 2
        return sorted(set(steps + [self.rl_steps]))
    @property
    def prompts_per_step(self): return self.per_device_bsz * self.grad_accum // self.num_generations
    def save(self):
        Path(self.ckpt_dir).mkdir(parents=True, exist_ok=True)
        json.dump(asdict(self), open(f"{self.run_dir}/config.json", "w"), indent=2); return f"{self.run_dir}/config.json"

CFGS = {size: Config(size) for size in RUN_SIZES}
for c in CFGS.values():
    c.save()
    print(f"{c.model_size:>4} | {c.gpu_preset} | full FT, {c.master_dtype} master weights | {c.prompts_per_step} prompts x {c.num_generations} samples/step "
          f"({c.per_device_bsz} x {c.grad_accum} accum) | {c.rl_steps} steps, ckpts {c.checkpoint_steps} | eval n={c.n_eval} k={c.k_max} | {c.run_dir}")

In [ ]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed); os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    transformers.set_seed(seed)
seed_everything(SEED)

### 0.4 Shared module and worker scripts
`common.py` holds the prompt, answer extractor, data loader, majority vote and the reward functions, so the trainer, the probe and the evaluator grade with identical code. The three workers are plain scripts the notebook launches with `subprocess`.

In [ ]:
%%writefile common.py
import re, json
from collections import Counter
from datasets import load_dataset

SYSTEM = "Think step by step. Then give the final answer on its own line as: #### <number>"
NUM = r"-?\$?\d[\d,]*\.?\d*"

def normalize(s):
    if s is None: return None
    s = s.replace(",", "").replace("$", "").strip().rstrip(".")
    try:
        f = float(s); return str(int(f)) if f == int(f) else str(f)
    except ValueError:
        return None

def extract(text):
    """Last '#### n' if present, else the last number in the text (what a truncated trace yields)."""
    m = re.findall(r"####\s*(" + NUM + ")", text)
    if m: return normalize(m[-1])
    nums = re.findall(NUM, text)
    return normalize(nums[-1]) if nums else None

def gold(answer_field): return normalize(answer_field.split("####")[-1])

def to_prompt(ex):
    return {"prompt": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": ex["question"]}],
            "answer": gold(ex["answer"])}

def load_gsm8k(split, n=None, seed=0):
    ds = load_dataset("openai/gsm8k", "main", split=split).map(to_prompt).shuffle(seed=seed)
    return ds.select(range(n)) if n else ds

def maj(answers):
    c = Counter(a for a in answers if a is not None)
    return c.most_common(1)[0][0] if c else None

def ks_up_to(k_max):
    ks, k = [], 1
    while k <= k_max: ks.append(k); k *= 2
    return ks

def make_config(cls, **kw):
    """Build a TrainingArguments-derived config, dropping kwargs the installed transformers/trl no longer accept
    (e.g. transformers>=5.2 removed warmup_ratio). Prints what it dropped so nothing disappears silently."""
    import dataclasses
    valid = {f.name for f in dataclasses.fields(cls)}
    dropped = {k: v for k, v in kw.items() if k not in valid}
    if dropped: print(f"[compat] {cls.__name__} does not accept {dropped} in this version - dropped", flush=True)
    return cls(**{k: v for k, v in kw.items() if k in valid})

def completion_text(c):
    """TRL passes completions as plain strings (standard datasets) or as lists of chat messages (conversational)."""
    if isinstance(c, str): return c
    if isinstance(c, dict): return c.get("content", "")
    return c[-1]["content"] if c else ""

def reward_correct(completions, answer, **kw):
    return [1.0 if extract(completion_text(c)) == a else 0.0 for c, a in zip(completions, answer)]

def reward_format(completions, **kw):
    return [0.2 if re.search(r"####\s*" + NUM, completion_text(c)) else 0.0 for c in completions]

In [ ]:
%%writefile probe_group.py
# §1 worker: sample G chains for a few training problems with the BASE model and print GRPO group statistics.
import argparse, json, numpy as np
from vllm import LLM, SamplingParams
from common import load_gsm8k, extract
p = argparse.ArgumentParser(); p.add_argument("--cfg", required=True); p.add_argument("--n", type=int, default=8); p.add_argument("--out", required=True)
a = p.parse_args(); cfg = json.load(open(a.cfg))
ds = load_gsm8k("train", n=a.n, seed=cfg["seed"])
llm = LLM(model=cfg["model_id"], dtype="bfloat16", seed=cfg["seed"], gpu_memory_utilization=0.6, max_model_len=cfg["max_prompt_len"] + cfg["max_completion_len"] + 64)
tok = llm.get_tokenizer()
prompts = [tok.apply_chat_template(ex["prompt"], tokenize=False, add_generation_prompt=True) for ex in ds]
outs = llm.generate(prompts, SamplingParams(n=cfg["num_generations"], temperature=cfg["temperature"], max_tokens=cfg["max_completion_len"], seed=cfg["seed"]))
rows = []
for ex, out in zip(ds, outs):
    r = np.array([1.0 if extract(s.text) == ex["answer"] else 0.0 for s in out.outputs])
    adv = (r - r.mean()) / (r.std() + 1e-4)
    rows.append({"gold": ex["answer"], "rewards": r.astype(int).tolist(), "group_mean": float(r.mean()),
                 "adv_correct": None if r.max() == 0 else float(adv[r == 1][0]), "mean_len": int(np.mean([len(s.token_ids) for s in out.outputs]))})
json.dump({"rows": rows, "sample": outs[0].outputs[0].text}, open(a.out, "w"))
print("probe done")

In [ ]:
%%writefile filter_prompts.py
# v3 worker: screen training prompts with the BASE model; keep those with 0 < pass-rate <= filter_max.
# All-correct groups have zero advantage (nothing to learn); all-wrong groups have zero advantage too (no success to reinforce).
import argparse, json, numpy as np
from vllm import LLM, SamplingParams
from common import load_gsm8k, extract
p = argparse.ArgumentParser(); p.add_argument("--cfg", required=True); p.add_argument("--out", required=True); a = p.parse_args()
cfg = json.load(open(a.cfg)); G = 8
ds = load_gsm8k("train", seed=cfg["seed"]).select(range(cfg["filter_n"]))
llm = LLM(model=cfg["model_id"], dtype="bfloat16", seed=cfg["seed"], gpu_memory_utilization=0.85, max_model_len=cfg["max_prompt_len"] + cfg["max_completion_len"] + 64, enable_prefix_caching=True)
tok = llm.get_tokenizer()
prompts = [tok.apply_chat_template(ex["prompt"], tokenize=False, add_generation_prompt=True) for ex in ds]
outs = llm.generate(prompts, SamplingParams(n=G, temperature=cfg["temperature"], max_tokens=cfg["max_completion_len"], seed=cfg["seed"]))
rates = np.array([np.mean([extract(s.text) == ex["answer"] for s in out.outputs]) for ex, out in zip(ds, outs)])
keep = [int(i) for i, r in enumerate(rates) if 0 < r <= cfg["filter_max"]]
hist = {f"{b/8:.3f}": int((rates == b / 8).sum()) for b in range(9)}
json.dump({"keep": keep, "rates_hist": hist, "n_screened": len(rates), "filter_max": cfg["filter_max"]}, open(a.out, "w"))
print(f"screened {len(rates)} prompts | pass-rate histogram (8 samples): {hist} | kept {len(keep)} ({len(keep)/len(rates):.0%})")

In [ ]:
%%writefile train_grpo.py
# §2 worker: GRPO, full fine-tuning with fp32 master weights, log-spaced bf16 checkpoints.
# Resumable: exits immediately if the final checkpoint exists.
import argparse, json, os, sys
from pathlib import Path
p = argparse.ArgumentParser(); p.add_argument("--cfg", required=True); a = p.parse_args()
cfg = json.load(open(a.cfg))
import torch
from transformers import TrainerCallback, set_seed
from trl import GRPOConfig, GRPOTrainer
from common import load_gsm8k, reward_correct, reward_format, make_config

def ckpt_steps(c):
    steps, s = [], max(10, c["rl_steps"] // 16)
    while s < c["rl_steps"]: steps.append(s); s *= 2
    return sorted(set(steps + [c["rl_steps"]]))
STEPS = ckpt_steps(cfg); CK = cfg["ckpt_dir"]; MASTER = getattr(torch, cfg["master_dtype"])
if (Path(CK) / f"step{cfg['rl_steps']}" / "config.json").exists():
    print("final checkpoint exists — skipping training"); sys.exit(0)

set_seed(cfg["seed"])
train_ds = load_gsm8k("train", seed=cfg["seed"])
flt = Path(cfg["run_dir"]) / "train_filter.json"
if cfg.get("filter_prompts") and flt.exists():
    keep = json.load(open(flt))["keep"]; train_ds = train_ds.select(keep)
    print(f"training on {len(train_ds)} difficulty-filtered prompts (0 < base pass-rate <= {cfg['filter_max']})", flush=True)

class MasterWeightsCheck(TrainerCallback):
    # fail before the first step if the optimizer would be updating anything but MASTER_DTYPE weights, or not all of them
    def on_train_begin(self, args, state, control, model=None, **kw):
        dts = {p.dtype for p in model.parameters()}; n_all = sum(p.numel() for p in model.parameters())
        n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
        assert dts == {MASTER}, f"parameters are {dts}, expected {MASTER}"
        assert n_train == n_all, f"only {n_train}/{n_all} parameters are trainable"
        print(f"master weights: {MASTER} | trainable {n_train/1e9:.3f}B of {n_all/1e9:.3f}B", flush=True)

class LogSpacedCheckpoint(TrainerCallback):
    # bf16 on disk: what vLLM serves, half the Drive space. Written to a temp folder and renamed, so a cut-off save never looks finished.
    def on_step_end(self, args, state, control, model=None, **kw):
        if state.global_step not in STEPS: return
        path = f"{CK}/step{state.global_step}"; tmp = path + ".tmp"
        tied = getattr(model.config, "tie_word_embeddings", False)
        sd = {k: v.detach().to(device="cpu", dtype=torch.bfloat16) for k, v in model.state_dict().items() if not (tied and k == "lm_head.weight")}
        model.save_pretrained(tmp, state_dict=sd, safe_serialization=True)
        c = json.load(open(f"{tmp}/config.json")); c["dtype"] = c["torch_dtype"] = "bfloat16"; json.dump(c, open(f"{tmp}/config.json", "w"), indent=2)
        os.replace(tmp, path); print(f"\n[ckpt] {path}", flush=True)

args = make_config(GRPOConfig,
    output_dir=f"{cfg['run_dir']}/trainer", seed=cfg["seed"], data_seed=cfg["seed"],
    learning_rate=cfg["learning_rate"], lr_scheduler_type="cosine", warmup_steps=max(1, int(0.05 * cfg["rl_steps"])),
    per_device_train_batch_size=cfg["per_device_bsz"], gradient_accumulation_steps=cfg["grad_accum"],
    num_generations=cfg["num_generations"], max_prompt_length=cfg["max_prompt_len"], max_completion_length=cfg["max_completion_len"],
    temperature=cfg["temperature"], beta=cfg["beta"], max_steps=cfg["rl_steps"], loss_type=cfg.get("loss_type", "grpo"),
    bf16=True, gradient_checkpointing=True, logging_steps=5, save_strategy="no", report_to="none",
    use_vllm=cfg["use_vllm"], vllm_mode="colocate", vllm_gpu_memory_utilization=cfg["vllm_util"],
    model_init_kwargs={"dtype": cfg["master_dtype"]},   # fp32 master weights; bf16=True autocasts the forward and backward
)
trainer = GRPOTrainer(model=cfg["model_id"], reward_funcs=[reward_correct, reward_format], args=args,
                      train_dataset=train_ds, callbacks=[MasterWeightsCheck(), LogSpacedCheckpoint()])
print(f"training {cfg['rl_steps']} steps | checkpoints at {STEPS}", flush=True)
trainer.train()
json.dump(trainer.state.log_history, open(f"{cfg['run_dir']}/log_history.json", "w"))
print("training done")

In [ ]:
%%writefile eval_ckpt.py
# §3 worker: one vLLM pass -> k samples per problem -> maj@k, pass@k, truncated-budget accuracy, lengths.
import argparse, json, time
from pathlib import Path
from vllm import LLM, SamplingParams
from common import load_gsm8k, extract, maj, ks_up_to
p = argparse.ArgumentParser()
for f in ["--model", "--tokenizer", "--out", "--tag", "--budgets"]: p.add_argument(f, required=True)
p.add_argument("--split", default="test"); p.add_argument("--n", type=int, required=True); p.add_argument("--k", type=int, required=True)
p.add_argument("--max_tokens", type=int, required=True); p.add_argument("--temperature", type=float, default=0.7)
p.add_argument("--seed", type=int, default=42); p.add_argument("--gpu_util", type=float, default=0.85); p.add_argument("--save_texts", action="store_true")
a = p.parse_args(); t0 = time.time()
ds = load_gsm8k(a.split, n=a.n, seed=a.seed)
llm = LLM(model=a.model, tokenizer=a.tokenizer, dtype="bfloat16", seed=a.seed, gpu_memory_utilization=a.gpu_util,
          max_model_len=512 + a.max_tokens, enable_prefix_caching=True)
tok = llm.get_tokenizer()
prompts = [tok.apply_chat_template(ex["prompt"], tokenize=False, add_generation_prompt=True) for ex in ds]
golds = [ex["answer"] for ex in ds]
outs = llm.generate(prompts, SamplingParams(n=a.k, temperature=a.temperature, top_p=0.95, max_tokens=a.max_tokens, seed=a.seed))
budgets = [int(b) for b in a.budgets.split(",")]; ks = ks_up_to(a.k)
per = []; lens = []; trunc = 0
for out, g in zip(outs, golds):
    ss = out.outputs; lens += [len(s.token_ids) for s in ss]; trunc += sum(len(s.token_ids) >= a.max_tokens for s in ss)
    per.append({"gold": g, "answers": [extract(s.text) for s in ss], "lens": [len(s.token_ids) for s in ss],
                "seq": {b: extract(tok.decode(ss[0].token_ids[:b])) for b in budgets},
                "texts": [s.text for s in ss] if a.save_texts else None})
per_prob_acc = [sum(y == x["gold"] for y in x["answers"]) / a.k for x in per]   # per-problem pass rate over all k samples
res = {"tag": a.tag, "model": a.model, "n": len(golds), "k_max": a.k, "max_tokens": a.max_tokens,
       "mean_acc": float(sum(per_prob_acc) / len(per)),                                   # pass@1 estimated from k samples (low-variance)
       "mean_acc_se": float((sum((v - sum(per_prob_acc) / len(per)) ** 2 for v in per_prob_acc) / (len(per) - 1)) ** 0.5 / len(per) ** 0.5),
       "maj_at_k": {k: sum(maj(x["answers"][:k]) == x["gold"] for x in per) / len(per) for k in ks},
       "pass_at_k": {k: sum(any(y == x["gold"] for y in x["answers"][:k]) for x in per) / len(per) for k in ks},
       "seq_at_budget": {b: sum(x["seq"][b] == x["gold"] for x in per) / len(per) for b in budgets},
       "mean_len": sum(lens) / len(lens), "frac_truncated": trunc / len(lens), "seconds": round(time.time() - t0, 1)}
Path(a.out).parent.mkdir(parents=True, exist_ok=True); json.dump(res, open(a.out, "w"), indent=1)
if a.save_texts:
    for x in per: x.pop("seq")
    json.dump(per, open(a.out.replace(".json", "_texts.json"), "w"))
print(json.dumps({k: v for k, v in res.items() if k in ("tag", "n", "mean_acc", "mean_acc_se", "mean_len", "frac_truncated", "seconds")}))
print("maj@k", res["maj_at_k"]); print("seq@budget", res["seq_at_budget"])

### 0.5 Orchestration helpers

In [ ]:
def run(cmd, log_path=None):
    # stream a worker's output into the notebook (and optionally a log file); raise on failure
    print(">", cmd, flush=True)
    proc = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail, f = [], (open(log_path, "a") if log_path else None)
    for line in proc.stdout:
        tail.append(line); tail = tail[-60:]
        if f: f.write(line)
        if "reward_correct" in line:                       # TRL metric dict: print the parts that matter, in full
            import re as _re
            keep = {k: v for k, v in _re.findall(r"'([^']+)': '([^']+)'", line) if k in ("rewards/reward_correct/mean", "completions/mean_length", "entropy", "epoch")}
            print(keep, flush=True)
        elif any(t in line for t in ("[ckpt]", "training on", "master weights", "screened", "done", "Error", "error", "maj@k", "seq@", "mean_acc")):
            print(line.rstrip()[-220:], flush=True)
    proc.wait()
    if f: f.close()
    if proc.returncode != 0:
        print("".join(tail)); raise RuntimeError(f"worker failed (exit {proc.returncode}): {cmd.split()[1]}")

def run_probe(cfg, n=8):
    out = f"{cfg.run_dir}/probe.json"
    if not Path(out).exists():
        run(f"{sys.executable} probe_group.py --cfg {cfg.run_dir}/config.json --n {n} --out {out}")
    return json.load(open(out))

def run_filter(cfg):
    out = f"{cfg.run_dir}/train_filter.json"
    if cfg.filter_prompts and not Path(out).exists():
        run(f"{sys.executable} filter_prompts.py --cfg {cfg.run_dir}/config.json --out {out}")
    return json.load(open(out)) if Path(out).exists() else None

def run_train(cfg):
    run(f"{sys.executable} train_grpo.py --cfg {cfg.run_dir}/config.json", log_path=f"{cfg.run_dir}/train.log")

def materialize(cfg, step):
    # path vLLM can load: the base model for step 0, the bf16 checkpoint otherwise (full fine-tuning: nothing to merge)
    return cfg.model_id if step == 0 else f"{cfg.ckpt_dir}/step{step}"

FAMILY = {"embed_tokens": "embedding", "q_proj": "attention", "k_proj": "attention", "v_proj": "attention", "o_proj": "attention",
          "gate_proj": "MLP", "up_proj": "MLP", "down_proj": "MLP", "norm": "norm"}
def family(key): return next((f for k, f in FAMILY.items() if k in key), "other")

def weight_movement(cfg):
    """Every checkpoint against the base, tensor by tensor, as stored (bf16): the share of weights that differ,
    and ||Δw|| / ||w||, per layer family and overall. CPU only; cached in the run's folder."""
    out = Path(cfg.run_dir) / "weight_movement.json"
    if out.exists(): return {int(k): v for k, v in json.load(open(out)).items()}
    from huggingface_hub import snapshot_download
    from safetensors import safe_open
    def index(d):
        return {k: (str(f), k) for f in sorted(Path(d).glob("*.safetensors")) for k in safe_open(str(f), "pt").keys()}
    def get(ref): return safe_open(ref[0], "pt").get_tensor(ref[1]).float()
    B = index(snapshot_download(cfg.model_id, allow_patterns=["*.safetensors", "*.json"]))
    res = {}
    for step in cfg.checkpoint_steps[1:]:
        T = index(f"{cfg.ckpt_dir}/step{step}")
        missing = set(T) - set(B); assert not missing, f"checkpoint tensors not in the base: {sorted(missing)[:5]}"
        acc = {}
        for k in T:
            t, b = get(T[k]), get(B[k]); d = t - b
            for fam in (family(k), "all"):
                a = acc.setdefault(fam, [0, 0, 0.0, 0.0])
                a[0] += int((d != 0).sum()); a[1] += d.numel(); a[2] += float((d * d).sum()); a[3] += float((b * b).sum())
        res[step] = {f: {"frac_changed": a[0] / a[1], "rel_norm": (a[2] / a[3]) ** 0.5, "n": a[1]} for f, a in acc.items()}
        print(f"step {step}: {res[step]['all']['frac_changed']:.1%} of weights differ from the base | ||Δw||/||w|| {res[step]['all']['rel_norm']:.2e}")
    json.dump(res, open(out, "w"), indent=1); return res

def run_eval(cfg, model_path, tag, split="test", n=None, k=None, save_texts=False, seed=None):
    out = f"{cfg.run_dir}/eval/{tag}.json"
    if Path(out).exists():
        print(f"[cached] {out}")
    else:
        run(f"{sys.executable} eval_ckpt.py --model {model_path} --tokenizer {cfg.model_id} --out {out} --tag {tag} --split {split} "
            f"--n {n or cfg.n_eval} --k {k or cfg.k_max} --max_tokens {cfg.eval_max_tokens} --budgets {','.join(map(str, cfg.budgets))} "
            f"--temperature {cfg.eval_temperature} --seed {seed or cfg.seed} {'--save_texts' if save_texts else ''}")
    return json.load(open(out))

# JSON turns int keys into strings; these accessors accept either
getk = lambda r, k: r["maj_at_k"].get(str(k), r["maj_at_k"].get(k))
getp = lambda r, k: r["pass_at_k"].get(str(k), r["pass_at_k"].get(k))
getb = lambda r, b: r["seq_at_budget"].get(str(b), r["seq_at_budget"].get(b))
from common import ks_up_to, extract, maj, load_gsm8k
# extractor regression tests — cheap insurance before spending GPU hours on this reward
for t, w in {"so the answer is #### 1,250": "1250", "#### $18.00": "18", "total = 42 so": "42", "none": None, "#### 3\nwait #### 5": "5"}.items():
    assert extract(t) == w, (t, extract(t), w)
print("helpers ready; extractor OK")

## 1. The loop: the group grades itself
For a single problem we sample *G* chains from the **base** model, grade each with a program against the answer key, and normalise within the group. A correct chain whose siblings failed gets a big push; a correct chain whose siblings also succeeded gets almost none. Groups that are all-0 or all-1 teach nothing — the task must sit in the model's zone of partial competence. If `group_mean` is 0 on every row, fix the prompt/extractor before training.

In [ ]:
for size, cfg in CFGS.items():
    pr = run_probe(cfg)
    print(f"\n=== {size} ({cfg.model_id}) ===")
    display(pd.DataFrame(pr["rows"]).round(2))
    print("one sampled chain:\n", pr["sample"][:700])

## 2. Axis 1 — RL lifts the curve
GRPO on GSM8K train, in a subprocess, with log-spaced checkpoints. The reward never mentions length; watch the right-hand panel anyway.

**v3: difficulty filtering first.** The base model is screened on `FILTER_N` training prompts with 8 samples each; only prompts with pass-rate in (0, `FILTER_MAX`] are kept. A group that is all-correct or all-wrong has zero advantage for every member — the step is spent but nothing is learned. The histogram printed below tells you how much of GSM8K was already "solved" by the base model.

In [ ]:
for size, cfg in CFGS.items():
    print(f"\n######## screening prompts for {size} ########")
    f = run_filter(cfg)
    if f: print(f"kept {len(f['keep'])}/{f['n_screened']} | base pass-rate histogram: {f['rates_hist']}")
for size, cfg in CFGS.items():
    print(f"\n######## training {size} ########"); t0 = time.time()
    run_train(cfg)
    print(f"{size}: {(time.time()-t0)/60:.1f} min")

In [ ]:
def plot_training(cfg):
    log = json.load(open(f"{cfg.run_dir}/log_history.json"))
    def series(keys):
        for key in keys:
            pts = [(r["step"], r[key]) for r in log if key in r]
            if pts: return list(zip(*pts))
        return [], []
    s1, r1 = series(["rewards/reward_correct/mean", "rewards/reward_correct"])
    s2, l2 = series(["completions/mean_length", "completion_length"])
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(s1, r1); ax[0].set(xlabel="RL step", ylabel="train reward (fraction correct)", title=f"{cfg.model_size}: training signal")
    ax[1].plot(s2, l2, color="C1"); ax[1].set(xlabel="RL step", ylabel="mean completion tokens", title="Length emerges (nothing rewards it)")
    for a in ax: a.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(f"{cfg.run_dir}/fig_training.png", dpi=130); plt.show()
for cfg in CFGS.values():
    if Path(f"{cfg.run_dir}/log_history.json").exists(): plot_training(cfg)

## 2b. Did the weights move?
Every checkpoint against the base, tensor by tensor, as stored in bf16.

In [ ]:
FAM_COLORS = {"attention": "#2a78d6", "MLP": "#eb6834", "embedding": "#1baf7a", "norm": "#eda100"}
MOVEMENT = {}
for size, cfg in CFGS.items():
    mv = MOVEMENT[size] = weight_movement(cfg)
    steps = sorted(mv)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for fam, col in FAM_COLORS.items():
        if fam in mv[steps[-1]]:
            ax[0].plot(steps, [mv[s][fam]["frac_changed"] * 100 for s in steps], "o-", color=col, label=fam)
            ax[1].plot(steps, [mv[s][fam]["rel_norm"] for s in steps], "o-", color=col, label=fam)
    for a in ax:
        a.plot(steps, [mv[s]["all"]["frac_changed"] * 100 if a is ax[0] else mv[s]["all"]["rel_norm"] for s in steps], "s--", color="#0b0b0b", label="all weights")
        a.set_xscale("log"); a.set_xticks(steps); a.set_xticklabels(map(str, steps)); a.grid(alpha=.3); a.set_xlabel("RL step (log)")
    ax[0].set(ylabel="weights that differ from the base (%)", ylim=(0, 105), title=f"{size}: how many weights moved")
    ax[1].set(ylabel="||Δw|| / ||w||", yscale="log", title="How far they moved"); ax[1].legend(fontsize=8)
    plt.tight_layout(); plt.savefig(f"{cfg.run_dir}/fig_weight_movement.png", dpi=130); plt.show()
    display(pd.DataFrame({f: {"weights changed": v["frac_changed"], "||Δw||/||w||": v["rel_norm"], "params (M)": v["n"] / 1e6}
                          for f, v in mv[steps[-1]].items()}).T.round(5))

## 3. Axis 2 — thinking slides along the curve
Every checkpoint is evaluated on the same fixed test subset with the same seed. One vLLM pass yields `k_max` samples per problem; the parallel axis (majority vote over the first *k*) and the sequential axis (first trace truncated at a token budget) are both read from those samples.

In [ ]:
RESULTS = {}
for size, cfg in CFGS.items():
    print(f"\n######## evaluating {size} ########")
    res = {}
    for step in cfg.checkpoint_steps:
        res[step] = run_eval(cfg, materialize(cfg, step), tag=f"step{step}")
    json.dump(res, open(f"{cfg.run_dir}/results.json", "w"), indent=1); RESULTS[size] = res
    tbl = pd.DataFrame({s: {"pass@1 (mean of k)": r.get("mean_acc", getk(r, 1)), "±SE": r.get("mean_acc_se", float("nan")), "maj@1 (first sample)": getk(r, 1),
                            f"maj@{cfg.k_max}": getk(r, cfg.k_max), f"pass@{cfg.k_max}": getp(r, cfg.k_max),
                            "mean len": round(r["mean_len"]), "truncated": round(r["frac_truncated"], 3)} for s, r in res.items()}).T
    tbl.index.name = "RL step"; print(size); display(tbl.round(3))

## 4. Both axes
**Left:** Axis 1 at one sample. **Middle:** Axis 2 (parallel) per checkpoint — later checkpoints start higher, and their slope says whether coaching makes extra exam-minutes more productive. **Right:** the substitution surface — follow a colour band diagonally: fewer RL steps + more samples ≈ more RL steps + fewer samples.

In [ ]:
def plot_two_axis(cfg, res):
    steps, ks = cfg.checkpoint_steps, ks_up_to(cfg.k_max); x1 = [s if s > 0 else steps[1] / 2 for s in steps]
    se = lambda p: np.sqrt(p * (1 - p) / cfg.n_eval); cmap = plt.get_cmap("Blues"); col = lambda i: cmap(0.35 + 0.65 * i / max(1, len(steps) - 1))
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.6)); fig.suptitle(f"{cfg.model_size} · GSM8K · n={cfg.n_eval} · seed {cfg.seed}", y=1.02)
    y1 = [res[s].get("mean_acc", getk(res[s], 1)) for s in steps]; e1 = [res[s].get("mean_acc_se", se(p)) for s, p in zip(steps, y1)]
    ax[0].errorbar(x1, y1, yerr=e1, fmt="o-", capsize=3); ax[0].set_xscale("log"); ax[0].set_xticks(x1); ax[0].set_xticklabels(map(str, steps))
    ax[0].set(xlabel="RL training steps (log)", ylabel=f"pass@1 (mean over {cfg.k_max} samples)", title="Axis 1: RL training compute")
    for i, s in enumerate(steps): ax[1].plot(ks, [getk(res[s], k) for k in ks], "o-", color=col(i), label=f"step {s}")
    ax[1].set_xscale("log", base=2); ax[1].set_xticks(ks); ax[1].set_xticklabels(ks); ax[1].legend(fontsize=8)
    ax[1].set(xlabel="samples k, majority vote (log)", ylabel="maj@k accuracy", title="Axis 2: test-time compute")
    grid = np.array([[getk(res[s], k) for k in ks] for s in steps])
    im = ax[2].imshow(grid, origin="lower", aspect="auto", cmap="Blues")
    for i in range(len(steps)):
        for j in range(len(ks)): ax[2].text(j, i, f"{grid[i,j]*100:.0f}", ha="center", va="center", fontsize=8, color="black" if grid[i,j] < grid.mean() else "white")
    ax[2].set(xticks=range(len(ks)), xticklabels=ks, yticks=range(len(steps)), yticklabels=steps, xlabel="samples k", ylabel="RL steps", title="Both axes (maj@k, %)")
    plt.colorbar(im, ax=ax[2], fraction=0.046)
    for a in ax[:2]: a.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(f"{cfg.run_dir}/fig_two_axis.png", dpi=130, bbox_inches="tight"); plt.show()

    fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
    for i, s in enumerate(steps): ax[0].plot(cfg.budgets, [getb(res[s], b) for b in cfg.budgets], "o-", color=col(i), label=f"step {s}")
    ax[0].set_xscale("log", base=2); ax[0].set_xticks(cfg.budgets); ax[0].set_xticklabels(cfg.budgets); ax[0].legend(fontsize=8)
    ax[0].set(xlabel="thinking budget (tokens, log)", ylabel="accuracy, 1 truncated sample", title="Axis 2 (sequential): accuracy vs. budget")
    ax[1].plot(x1, [res[s]["mean_len"] for s in steps], "s-", color="C1"); ax[1].set_xscale("log"); ax[1].set_xticks(x1); ax[1].set_xticklabels(map(str, steps))
    ax[1].set(xlabel="RL training steps (log)", ylabel="mean trace length (tokens)", title="RL buys longer traces on its own")
    for a in ax: a.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(f"{cfg.run_dir}/fig_sequential.png", dpi=130); plt.show()

for size, cfg in CFGS.items(): plot_two_axis(cfg, RESULTS[size])

## 5. Stacking on top
Answer-time methods still add to a trained reasoner, but they are alternatives, not a ladder, and each is paid per question. `pass@k` is the oracle: what a perfect re-ranker over *k* chains could reach — the gap between `maj@k` and `pass@k` is what a learned verifier would buy.

In [ ]:
def plot_stacking(cfg, res):
    final, K = cfg.checkpoint_steps[-1], cfg.k_max
    p1 = lambda r: r.get("mean_acc", getk(r, 1))
    bars = {"base, pass@1": p1(res[0]), f"RL step {final}, pass@1": p1(res[final]),
            f"RL step {final}, maj@{K}": getk(res[final], K), f"RL step {final}, pass@{K} (oracle)": getp(res[final], K)}
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.barh(list(bars), [v * 100 for v in bars.values()], color=["#9aa3ad", "#2a78d6", "#2a78d6", "#85b7eb"])
    for i, v in enumerate(bars.values()): ax.text(v * 100 + 1, i, f"{v*100:.1f}%", va="center")
    ax.set(xlim=(0, 105), xlabel="GSM8K accuracy (%)", title=f"{cfg.model_size}: one model, read four ways"); ax.invert_yaxis(); ax.grid(axis="x", alpha=.3)
    plt.tight_layout(); plt.savefig(f"{cfg.run_dir}/fig_stacking.png", dpi=130); plt.show()
for size, cfg in CFGS.items(): plot_stacking(cfg, RESULTS[size])

## 6. Model size picks the curve
Overlays the sizes finished in this run.

In [ ]:
runs = {}
for f in sorted(glob.glob(f"{RUN}/*/results.json")):
    c = json.load(open(Path(f).parent / "config.json")); r = json.load(open(f))
    runs[f"{c['model_size']} · {c['gpu_preset']} · seed {c['seed']}"] = (c, {int(k): v for k, v in r.items() if k.isdigit()})
print("runs found:", list(runs))
if runs:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
    for lab, (c, r) in runs.items():
        st = sorted(r); xs = [s if s > 0 else st[1] / 2 for s in st]
        ax[0].plot(xs, [r[s].get("mean_acc", getk(r[s], 1)) for s in st], "o-", label=lab)
        kk = ks_up_to(c["k_max"]); ax[1].plot(kk, [getk(r[st[-1]], k) for k in kk], "o-", label=f"{lab} @ step {st[-1]}")
    ax[0].set_xscale("log"); ax[0].set(xlabel="RL steps (log)", ylabel="pass@1 (mean over k samples)", title="Axis 1 across model sizes")
    ax[1].set_xscale("log", base=2); ax[1].set(xlabel="samples k (log)", ylabel="maj@k", title="Axis 2 across model sizes (final checkpoint)")
    for a in ax: a.grid(alpha=.3); a.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(f"{RUN}/fig_cross_size.png", dpi=130); plt.show()

## 7. Record the run
Writes `derived.json`, which marks the run finished: re-running the notebook after this starts a new run. The final checkpoint is the distillation notebook's `TEACHER_WEIGHTS`.

In [ ]:
derived = {"finished_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "knobs": KNOBS,
           "env": {"gpu": gpu.name, "torch": torch.__version__, "vllm": vllm.__version__, "trl": trl.__version__, "transformers": transformers.__version__},
           "sizes": {}}
for size, cfg in CFGS.items():
    res, final = RESULTS[size], cfg.checkpoint_steps[-1]
    derived["sizes"][size] = {
        "config": asdict(cfg), "final_checkpoint": f"{cfg.ckpt_dir}/step{final}",
        "pass_at_1": {s: r.get("mean_acc") for s, r in res.items()}, "pass_at_1_se": {s: r.get("mean_acc_se") for s, r in res.items()},
        "maj_at_k": {s: r["maj_at_k"] for s, r in res.items()}, "pass_at_k": {s: r["pass_at_k"] for s, r in res.items()},
        "seq_at_budget": {s: r["seq_at_budget"] for s, r in res.items()}, "mean_len": {s: r["mean_len"] for s, r in res.items()},
        "weight_movement": MOVEMENT[size]}
    print(f"{size}: pass@1 {res[0]['mean_acc']:.3f} -> {res[final]['mean_acc']:.3f} | "
          f"{MOVEMENT[size][final]['all']['frac_changed']:.1%} of weights moved | teacher for distillation: {cfg.ckpt_dir}/step{final}")
json.dump(derived, open(RUN / "derived.json.tmp", "w"), indent=1); os.replace(RUN / "derived.json.tmp", RUN / "derived.json")
print(f"{RUN}/derived.json")

## Reading the results honestly
- **Error bars.** One standard error on a proportion near 50% is ≈ 0.5/√n: 2.9 points at n=300, 3.5 at n=200. Differences inside that are noise; raise `n_eval` (eval-time only) before trusting a small effect.
- **Log-linearity.** With 5–6 checkpoints you can see "each doubling buys roughly the same number of points"; you cannot defend a fitted slope. The point is the *shape* and the *substitution*, not the coefficient.
- **Saturation.** `maj@k` flattens when the modal answer stops changing; `pass@k` keeps rising because it is an oracle.
- **Truncation.** If `frac_truncated` is high at the final checkpoint, traces are hitting `eval_max_tokens`; raise the budgets or the sequential curve is capped by you, not the model.
- **Reward hacking.** If train reward climbs but test `maj@1` doesn't, read traces from the final checkpoint (`run_eval(..., save_texts=True)`). The extractor takes the *last* `####`, which blunts the most common hack (emitting several answers).

### Troubleshooting
| Symptom | Fix |
|---|---|
| `undefined symbol: ncclCommResume` or any `libtorch_cuda.so` error | torch and `nvidia-nccl-cu12` disagree. Delete `/content/.two_axis_env_ok`, **Runtime → Disconnect and delete runtime**, re-run 0.1. Never `pip install torch==…` yourself. |
| 0.1 "crashes" after installing | Expected — it restarts the runtime. Re-run from the top. |
| `TypeError: GRPOConfig.__init__() got an unexpected keyword argument` | an API rename in transformers/trl. `make_config` in `common.py` drops unknown arguments and prints them; if a *required* one is renamed, check the current TRL GRPOConfig docs |
| vLLM colocate fails inside the trainer | set `USE_VLLM=False` (3–4× slower; halve `rl_steps`) or pin `trl`/`vllm` to the pair in TRL's GRPO docs |
| `parameters are {torch.bfloat16}, expected torch.float32` | this TRL ignored `model_init_kwargs`; load the model yourself in `train_grpo.py` (`AutoModelForCausalLM.from_pretrained(id, dtype=torch.float32)`) and pass the object as `model=` |
| OOM during training | the fp32 optimizer state is the new cost. Add `vllm_enable_sleep_mode=True` to the `GRPOConfig` call in `train_grpo.py` (vLLM offloads its weights during the optimizer step); or halve `bsz` and double `accum` in the preset (prompts/step unchanged); or lower `vllm_util` to 0.2 |
| train reward ≈ 0 for 30+ steps | check §1: `group_mean` must be > 0 on most rows |
| eval worker OOM | something else holds the GPU: `!nvidia-smi`; the kernel itself should hold nothing between stages |
| Colab disconnect | set `MOUNT_DRIVE=True` and re-run from the top: the unfinished run resumes, training skips if its final checkpoint exists (a cut-off run restarts training from step 0), eval is kept per checkpoint |